In [1]:
# Data manipulation and preprocessing
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, train_test_split, cross_val_score, cross_val_predict
from sklearn.preprocessing import StandardScaler

# Classification models
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
import lightgbm as lgb
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.neural_network import MLPClassifier
from sklearn.utils.class_weight import compute_sample_weight
import numpy as np
# Evaluation metrics
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
                             precision_score, recall_score, f1_score, make_scorer, roc_curve, accuracy_score)

# Feature selection
from sklearn.feature_selection import SelectKBest, chi2, RFE

# Data balancing (if necessary)
from imblearn.over_sampling import SMOTE

# Handling warnings
import warnings
warnings.filterwarnings("ignore")

/usr/local/lib/python3.10/dist-packages/dask/dataframe/__init__.py:42: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)


In [2]:
# Load the dataset

df_filtered = pd.read_csv("combined_df.csv")
df_filtered.head()

,resource_type_x,id_x,study_id_x,condition,disease_comment,age_at_diagnosis,age,height,weight,gender,...,Swelling,Daytime sleepiness,Insomnia,Intense vivid dreams,Acting out during dreams,Restless legs,Rencoded,condition_Healthy,condition_Other_Disorders,condition_Parkinson's
0,patient,1,PADS,Healthy,-,56,56,173,78,male,...,0,0,0,0,0,0,Healthy,True,False,False
1,patient,2,PADS,Other Movement Disorders,Left-Sided resting tremor and hypokinesia with...,69,81,193,104,male,...,1,1,1,0,1,0,Other_Disorders,False,True,False
2,patient,3,PADS,Healthy,-,45,45,170,78,female,...,0,0,0,0,0,0,Healthy,True,False,False
3,patient,4,PADS,Parkinson's,IPS akinetic-rigid type,63,67,161,90,female,...,1,1,1,0,0,1,Parkinson's,False,False,True
4,patient,5,PADS,Parkinson's,IPS tremordominant type,65,75,172,86,male,...,1,1,1,1,1,1,Parkinson's,False,False,True


In [3]:
# Select relevant symptom columns
symptom_columns = ['Dribbling', 'Swallowing', 'Vomiting', 'Constipation', 'Bowel inconsistence',
                   'Bowel emptying incomplete', 'Urgency', 'Nocturia', 'Pains', 'Weight',
                   'Sweating', 'Diplopia', 'Remembering', 'Loss of interest', 'Concentrating',
                   'Taste/smelling', 'Hallucinations', 'Delusions', 'Sad, blues', 'Anxiety',
                   'Sex drive', 'Sex difficulty', 'Dizzy', 'Falling', 'Swelling',
                   'Daytime sleepiness', 'Insomnia', 'Intense vivid dreams',
                   'Acting out during dreams', 'Restless legs']

In [5]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.7/98.7 MB 9.1 MB/s eta 0:00:00


In [8]:
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler, PolynomialFeatures
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
import lightgbm as lgb

# Define symptom columns (as previously defined)
symptom_columns = ['Dribbling', 'Swallowing', 'Vomiting', 'Constipation', 'Bowel inconsistence',
                   'Bowel emptying incomplete', 'Urgency', 'Nocturia', 'Pains', 'Weight',
                   'Sweating', 'Diplopia', 'Remembering', 'Loss of interest', 'Concentrating',
                   'Taste/smelling', 'Hallucinations', 'Delusions', 'Sad, blues', 'Anxiety',
                   'Sex drive', 'Sex difficulty', 'Dizzy', 'Falling', 'Swelling',
                   'Daytime sleepiness', 'Insomnia', 'Intense vivid dreams',
                   'Acting out during dreams', 'Restless legs']

# Features and target
X = df_filtered[symptom_columns]
y = df_filtered['Rencoded']

# Encode the target labels as integers
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)  # Encodes 'Healthy', 'Other_Disorders', 'Parkinson's' as 0, 1, 2

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42)

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Define the classification models
models = {
    'KNN': KNeighborsClassifier(n_neighbors=3),
    'Naive Bayes': GaussianNB(),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(n_estimators=100),
    "Gradient Boosting": GradientBoostingClassifier(),
    "AdaBoost": AdaBoostClassifier(algorithm="SAMME"),
    "XGB": XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3),
    "Extra Trees": ExtraTreesClassifier(n_estimators=100),
    "LightGBM": lgb.LGBMClassifier(),
    'SVM': SVC(),
    'MLP': MLPClassifier(max_iter=2000, random_state=42),
    'CatBoost': CatBoostClassifier(iterations=100, learning_rate=0.1, depth=3, verbose=0, random_state=42),
    'Polynomial Regression': Pipeline([('poly', PolynomialFeatures(degree=2)), ('model', LogisticRegression(max_iter=1000))])
}

# Stratified K-Fold Cross-Validation
kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Function to evaluate model
def evaluate_model(model, X_train, y_train, X_test, y_test, class_labels):
    # Cross-validation
    fold_results = []
    for fold_idx, (train_index, val_index) in enumerate(kf.split(X_train, y_train)):
        # Use the indices to access the corresponding rows in the numpy arrays
        X_fold_train, X_fold_val = X_train[train_index], X_train[val_index]
        y_fold_train, y_fold_val = y_train[train_index], y_train[val_index]

        model.fit(X_fold_train, y_fold_train)
        y_fold_pred = model.predict(X_fold_val)

        # Classification report for each fold
        report = classification_report(y_fold_val, y_fold_pred, output_dict=True, zero_division=0)
        fold_results.append(report)
        print(f"\nClassification Report for Fold {fold_idx + 1}:")
        print(classification_report(y_fold_val, y_fold_pred, target_names=class_labels, zero_division=0))

    # Averaged metrics across folds
    weighted_avg_precision = np.mean([result['weighted avg']['precision'] for result in fold_results])
    weighted_avg_recall = np.mean([result['weighted avg']['recall'] for result in fold_results])
    weighted_avg_f1 = np.mean([result['weighted avg']['f1-score'] for result in fold_results])
    print(f"\nWeighted Average (Training) for model:")
    print(f"Precision: {weighted_avg_precision:.2f}, Recall: {weighted_avg_recall:.2f}, F1-Score: {weighted_avg_f1:.2f}")

    # Test set evaluation
    model.fit(X_train, y_train)  # Fit on entire training set
    y_test_pred = model.predict(X_test)
    print("\nTest Set Evaluation:")
    print(classification_report(y_test, y_test_pred, target_names=class_labels, zero_division=0))
    print(f"\nConfusion Matrix for model (Testing):")
    print(confusion_matrix(y_test, y_test_pred))

# Run models and display results
class_labels = label_encoder.classes_  # Retrieve the original class names from the encoder
for name, model in models.items():
    print(f"\nEvaluating {name}...")
    evaluate_model(model, X_train_scaled, y_train, X_test_scaled, y_test, class_labels)


Streaming output truncated to the last 5000 lines.

        Healthy       0.71      0.83      0.77         6
Other_Disorders       0.22      0.22      0.22         9
    Parkinson's       0.71      0.68      0.70        22

       accuracy                           0.59        37
      macro avg       0.55      0.58      0.56        37
   weighted avg       0.59      0.59      0.59        37


Classification Report for Fold 9:
                 precision    recall  f1-score   support

        Healthy       0.44      0.67      0.53         6
Other_Disorders       0.50      0.22      0.31         9
    Parkinson's       0.83      0.91      0.87        22

       accuracy                           0.70        37
      macro avg       0.59      0.60      0.57        37
   weighted avg       0.69      0.70      0.68        37


Classification Report for Fold 10:
                 precision    recall  f1-score   support

        Healthy       0.40      0.33      0.36         6
Other_Disorders 